### FASE SCRUB

In [1]:
import pandas as pd
import numpy as np

# Configuración de archivos
ARCHIVO_ENTRADA = "../Obtain/encuestas_raw.csv"
ARCHIVO_SALIDA = "datos_Scrub.csv"
TEXTO_RELLENO = "No especifica"

# Diccionarios de mapeo
MAPA_COLUMNAS = {
    0: "timestamp", 
    1: "edad", 
    2: "genero", 
    3: "institucion", 
    4: "area_residencia", 
    5: "año_escolar", 
    6: "idiomas", 
    7: "edu_padres", 
    8: "ingreso_familiar", 
    9: "atividades_interes",
    10: "actividades_gustan_realizar", 
    11: "asignaturas", 
    12: "carreras", 
    13: "influencia_carrera", 
    14: "likert_apoyo_maestros", 
    15: "opinion_amigos", 
    16: "likert_apoyo_familia", 
    17: "obstaculos", 
    18: "sectores_interes", 
    19: "likert_optimismo", 
    20: "likert_estabilidad_eco", 
    21: "impacto_esperado", 
    22: "likert_percep_tecnologia", 
    23: "area_tecnol_importante", 
    24: "likert_prob_steam", 
    25: "nivel_informacion_steam"
}

DICCIONARIO_ORTOGRAFIA = {
    "Masculico": "Masculino",
    "Español e inglés": "Español e Inglés",
    "Españo,Ingles": "Español e Inglés"
}

# Diccionarios para ordinales y selección múltiple
MAPAS_ORDINALES = {
    "genero": {"Femenino": 0, "Masculino": 1},
     
    "institucion": {"VICENTE ANDA AGUIRRE": 0, "LICENCIADO RAFAEL FIALLOS": 1},
    
    "area_residencia": {"Rural": 0, "Urbana": 1, "No especifica": 2},
        
    "año_escolar": {"Primero de bachillerato": 1, "Segundo de bachillerato": 2, "Tercero de bachillerato": 3},
    
    "idiomas": {"Español": 1, "Español e Inglés": 2, "Otro": 3},
    
    "edu_padres": {"Primaria": 1, "Secundaria": 2, "Bachillerato": 3, "Universidad": 4, "Postgrado": 5},
    
    "ingreso_familiar": {"Menos de $500": 1, "$501 - $1000": 2, "$1001 - $2000": 3, "$2000 - $3000": 4, "Más de $3000": 5},
    
    "opinion_amigos": {"Muy negativo": 1, "Negativo": 2, "Neutro": 3, "Positivo": 4, "Muy positivo": 5},
    
    "obstaculos": {"Falta de recursos educativos": 1, "Falta de apoyo familiar": 2, "Dificultad en las materias": 3, "Otro": 4},
    
    "sectores_interes": {"Salud": 1, "Tecnología": 2, "Medio ambiente": 3, "Arte": 4, "Educación (T/S/A)": 5},
    
    "impacto_esperado": {"Mejorar la calidad de vida de las personas": 1, "Innovar y crear nuevas tecnologías": 2, "Contribuir a la sostenibilidad": 3, "Desarrollar nuevas ideas artísticas": 4, "No estoy seguro(a)": 5},
    
    "area_tecnol_importante": {"Salud": 1, "Energía": 2, "Educación": 3, "Transporte": 4, "Medio ambiente": 5},
    
    "nivel_informacion_steam": {"Nada informado/a": 1, "Poco informado/a": 2, "Neutro": 3, "Informado/a": 4, "Muy informado/a": 5}
}

MAPAS_MULTISELECT = {
    
    "atividades_interes": {
    "herramientas, máquinas y equipos"      : "q9_herramientas",
    "construyendo cosas físicas"          : "q9_construccion_fisica",
    "mecánica y automóviles"                : "q9_mecanica",
    "seres vivos como animales"             : "q9_seres_vivos",
    "ingeniería y cómo se utiliza"          : "q9_ingenieria",
    "Me interesa investigar fenómenos físicos":"q9_fenomenos_f",
    "ajedrez"                               : "q9_ajedrez",
    "ciencia ficción y avances científicos" : "q9_ciencia_ficcion",
    "misterios sin resolver"                : "q9_misterios",
    "novelas de detectives"                 : "q9_logica_deductiva",
    "Me interesa trabajar con materiales físicos"     : "q9_artes_plasticas",
    "arte y belleza visual"                 : "q9_arte_visual",
    "actividades artísticas como la música" : "q9_actividades_artisticas",
    },
    
    "actividades_gustan_realizar": {
    "Analizar datos para encontrar patrones": "q10_analisis_datos",    
    "Aprender sobre nuevos descubrimientos científicos": "q10_descubrimientos_cient",
    "Explorar nuevas tecnologías": "q10_explorar_tecnologia",
    "Resolver problemas usando herramientas y software": "q10_problemas_software",
    "Diseñar y construir estructuras": "q10_diseño_construccion",
    "Resolver problemas complejos de manera creativa": "q10_problemas_creativos",
    "Crear obras artísticas como pinturas": "q10_artes_visuales",
    "Compartir y recibir retroalimentación sobre tu arte": "q10_retroalim_arte",
    "Analizar datos y realizar interpretaciones estadísticas": "q10_estadistica",
    "Resolver problemas matemáticos": "q10_problemas_matematicos"
    },
    
    "asignaturas":{
    "Matemáticas"           : "q11_matematicas",
    "Biología"          : "q11_biologia",
    "Física"            : "q11_fisica",
    "Química"      : "q11_quimica",
    "Tecnología": "q11_tecnologia",
    "Arte": "q11_arte"
    },
    
    "carreras":{
    "Investigador/a científico/a": "q12_investigador",
    "Desarrollador/a de software": "q12_desarrollador",
    "Ingeniero/a civil": "q12_ingeniero_civil",
    "Artista digital": "q12_artista_digital",
    "Matemático/a o estadístico/a": "q12_matematico",
    "Otro": "q12_otro" 
    },
    
    
    "obstaculos": {
    "Falta de recursos educativos":"q17_Falta_recursos_educativos", "Falta de apoyo familiar":"q17_Falta_apoyo_familiar", "Dificultad en las materias":"q17_dificultad_materias", "Otro":"q17_Otro"
    },
    
    "sectores_interes": {"Salud":"q18_salud", "Tecnología":"q18_tecnologia", "Medio ambiente":"q18_medio_ambiente", "Arte":"q18_aete","Educación (T/S/A)":"q18_educacion",},
    
    "impacto_esperado":{"Mejorar la calidad de vida de las personas":"q21_mejorar_calidad_personas","Innovar y crear nuevas tecnologías":"q21_innovar_tecnologia" ,"Contribuir a la sostenibilidad":"q21_contribuir_sostenibilidad", "Desarrollar nuevas ideas artísticas":"q21_desarrollo_ideas_artisticas", "No estoy seguro(a)":"q21_no_estoy_seguro"},
    
    "area_tecnol_importante": {"Salud":"q23_salud", "Energía":"q23_energia", "Educación":"q23_educacion", "Transporte":"q23_transporte", "Medio ambiente":"q23_medio_ambiente"}
    
    } 


COLS_LIKERT = ['likert_apoyo_maestros', 'likert_apoyo_familia', 'likert_optimismo', 'likert_estabilidad_eco', 'likert_percep_tecnologia', 'likert_prob_steam']


# --- Inicio del script ---
print("Iniciando proceso de limpieza de datos...")

# Carga y estandarización inicial
df = pd.read_csv(ARCHIVO_ENTRADA, encoding='latin1')

# Renombrar columnas según el índice
df = df.rename(columns={df.columns[k]: 
    v for k, v in MAPA_COLUMNAS.items() if k < len(df.columns)})
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

# Limpiar texto invisible y espacios
df = df.replace(r'[\x7f]+', '', regex=True)
cols_texto = df.select_dtypes(include=['object']).columns
for col in cols_texto:
    df[col] = df[col].str.strip()

df.info()
df = df.replace(DICCIONARIO_ORTOGRAFIA)
print(f"Datos cargados. Total inicial: {len(df)} encuestas.")

# Filtrado de anomalías
df[COLS_LIKERT] = df[COLS_LIKERT].apply(pd.to_numeric, errors='coerce')

idx_edades = df[(df['edad'] < 14) | (df['edad'] > 20)].index
idx_duplicados = df[df.duplicated()].index
idx_sospechosos = df[df[COLS_LIKERT].std(axis=1) == 0].index
idx_vacios = df[(df.isnull().sum(axis=1) / len(df.columns)) > 0.40].index

filas_a_eliminar = set(idx_edades) | set(idx_duplicados) | set(idx_sospechosos) | set(idx_vacios)
df_limpio = df.drop(index=filas_a_eliminar).copy()
print(f"Se eliminaron {len(filas_a_eliminar)} encuestas inválidas o duplicadas.")

# Transformación y cálculo de nuevas variables
# Rellenar vacíos en texto
cols_texto = df_limpio.select_dtypes(include=['object', 'string']).columns
columnas_modificadas = 0

print("-" * 70)
print("Rellenando datos vacíos en columnas de texto con la MODA:")

for col in cols_texto:
    # Verificamos si la columna tiene datos nulos
    num_vacios = df_limpio[col].isnull().sum()
    if num_vacios > 0:
        # Calculamos la moda de la columna
        moda_col = df_limpio[col].mode()[0]
        # Rellenamos los valores nulos con esa moda
        df_limpio[col] = df_limpio[col].fillna(moda_col)
        
        columnas_modificadas += 1
        print(f" -> '{col}': {num_vacios} vacíos rellenados con '{moda_col}'")

print("-" * 70)
print(f"Se rellenaron los datos vacíos en {columnas_modificadas} columnas de texto usando la moda.")

# Mapeo ordinal
for col, mapa in MAPAS_ORDINALES.items():
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].astype(str).str.strip()
        df_limpio[f"{col}_num"] = df_limpio[col].map(mapa)

# Reglas de relleno específicas
#if "obstaculos_texto_num" in df_limpio.columns:
#    df_limpio["obstaculos_texto_num"] = df_limpio["obstaculos_texto_num"].fillna(4)
#if "impacto_esperado_num" in df_limpio.columns:
#    df_limpio["impacto_esperado_num"] = df_limpio["impacto_esperado_num"].fillna(6)


# --------------------------------------------------------------------------- #
# RELLENAR TODOS LOS DATOS FALTANTES CON LA MODA (TODO EL DATASET)
# --------------------------------------------------------------------------- #
print("-" * 70)
print("Imputando valores faltantes con la moda en todo el dataset...")

# Recorremos TODAS las columnas del dataframe limpio
for col in df_limpio.columns:
    num_vacios = df_limpio[col].isnull().sum()
    
    # Si la columna tiene al menos 1 dato vacío
    if num_vacios > 0:
        # Calculamos la moda de esa columna (tomamos el [0] en caso de empates)
        moda_columna = df_limpio[col].mode()[0]
        
        # Rellenamos los vacíos con la moda
        df_limpio[col] = df_limpio[col].fillna(moda_columna)
        
        # Imprimimos el registro para saber qué hizo el código
        print(f" -> Columna '{col}': {num_vacios} vacíos rellenados con '{moda_columna}'")

print("Todos los datos faltantes han sido reemplazados.")
print("-" * 70)

# --------------------------------------------------------------------------- #
# TRATAMIENTO DIFERENCIADO: CARRERAS (Múltiple) y OBSTÁCULOS (Única)
# --------------------------------------------------------------------------- #
print("-" * 70)
print("Estandarizando respuestas múltiples ('carreras') y únicas ('obstáculos')...")

# =========================================================================== #
# CASO 1: CARRERAS (Selección múltiple separada por comas)
# =========================================================================== #
carreras_oficiales = [
    "Investigador/a Cientifico/a",
    "Desarrollador/a de software",
    "Ingeniero/a civil",
    "Artista digital",
    "Matematico/a o estadistico/a",
    "Otro"
]

if "carreras" in df_limpio.columns:
    def procesar_carreras(respuesta):
        if pd.isna(respuesta): return respuesta
        
        # Separar por comas y limpiar espacios
        opciones = [op.strip() for op in str(respuesta).split(',')]
        
        # Revisar si hay alguna carrera que no esté en la lista oficial
        tiene_extra = False
        for op in opciones:
            if op != "" and op not in carreras_oficiales:
                tiene_extra = True
                break
                
        # Si hay algo extra y el estudiante no marcó "Otro", se lo añadimos
        if tiene_extra and "Otro" not in opciones:
            return str(respuesta) + ", Otro"
        return str(respuesta)

    df_limpio["carreras"] = df_limpio["carreras"].apply(procesar_carreras)
    print(" -> 'carreras' (Múltiple): Respuestas personalizadas anexadas como 'Otro'.")

# =========================================================================== #
# CASO 2: OBSTÁCULOS (Respuesta única + Variable numérica)
# =========================================================================== #
obstaculos_oficiales = [
    "Falta de recursos educativos",
    "Falta de apoyo familiar",
    "Dificultad en las materias",
    "Otro",
    "No especifica"
]

mapa_numerico_obstaculos = {
    "Falta de recursos educativos": 1,
    "Falta de apoyo familiar": 2,
    "Dificultad en las materias": 3,
    "Otro": 4,
    "No especifica": 5
}

if "obstaculos" in df_limpio.columns:
    # 1. Limpiar espacios
    df_limpio["obstaculos"] = df_limpio["obstaculos"].astype(str).str.strip()
    
    # 2. Reemplazar cualquier texto fuera de lista por "Otro"
    mascara_rebelde = ~df_limpio["obstaculos"].isin(obstaculos_oficiales)
    num_cambios = mascara_rebelde.sum()
    df_limpio.loc[mascara_rebelde, "obstaculos"] = "Otro"
    
    # 3. Crear la variable numérica mapeando los textos exactos
    df_limpio["obstaculos_texto_num"] = df_limpio["obstaculos"].map(mapa_numerico_obstaculos)
    
    print(f" -> 'obstaculos' (Única): {num_cambios} respuestas convertidas a 'Otro'.")
    print(" -> Columna 'obstaculos_texto_num' (1-5) creada correctamente.")

print("-" * 70)
    
# --------------------------------------------------------------------------- #
# # Transformación One-Hot (Usando funciones nativas de Pandas)
# --------------------------------------------------------------------------- # 

for col_fuente, opciones_map in MAPAS_MULTISELECT.items():
    if col_fuente in df_limpio.columns:
        for texto_opcion, nombre_col in opciones_map.items():
            # str.contains 
            df_limpio[nombre_col] = df_limpio[col_fuente].str.lower().str.contains(texto_opcion.lower(), na=False, regex=False).astype(int)
# --------------------------------------------------------------------------- #
# BINARIZACIÓN DE VARIABLES LIKERT Y ORDINALES
# --------------------------------------------------------------------------- #
print("-" * 70)
print("Binarizando variables Likert y ordinales...")

# Diccionario de umbrales. 
# Valor numérico indica a partir de qué número se considerará como "1" (Sí/Positivo).
UMBRALES_BINARIOS = {
    # Variables Likert (4 = De acuerdo, 5 = Muy de acuerdo -> 1)
    'likert_apoyo_maestros': 4,
    'likert_apoyo_familia': 4,
    'likert_optimismo': 4,
    'likert_estabilidad_eco': 4,
    'likert_percep_tecnologia': 4,
    'likert_prob_steam': 4,       # <-- Generará tu variable OBJETIVO
    
    # Variables ordinales (Deben usar el nombre de la columna que ya tiene '_num')
    'opinion_amigos_num': 4,          # 4 = Positivo, 5 = Muy positivo -> 1
    'nivel_informacion_steam_num': 4, # 4 = Informado, 5 = Muy informado -> 1
    
    # Ejemplo extra: Ingreso familiar
    # 'ingreso_familiar_num': 3       # Si quieres que ingresos mayores a $1000 sean 1
}

for col, umbral in UMBRALES_BINARIOS.items():
    if col in df_limpio.columns:
        nombre_binario = f"{col}_binario"
        # Si el valor es mayor o igual al umbral -> 1, de lo contrario -> 0
        df_limpio[nombre_binario] = (df_limpio[col] >= umbral).astype(int)

df_limpio.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8-sig")

print(f"Proceso finalizado con éxito.")
print(f"Dataset exportado a '{ARCHIVO_SALIDA}' ({len(df_limpio)} filas x {len(df_limpio.columns)} columnas).")

Iniciando proceso de limpieza de datos...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   edad                         316 non-null    int64  
 1   genero                       314 non-null    object 
 2   institucion                  316 non-null    object 
 3   area_residencia              310 non-null    object 
 4   año_escolar                  314 non-null    object 
 5   idiomas                      312 non-null    object 
 6   edu_padres                   316 non-null    object 
 7   ingreso_familiar             316 non-null    object 
 8   atividades_interes           316 non-null    object 
 9   actividades_gustan_realizar  316 non-null    object 
 10  asignaturas                  316 non-null    object 
 11  carreras                     316 non-null    object 
 12  influencia_carrera           311 non